# Fake Jobs – Exp 2: Enhanced Baseline-Modelle
- Gleiche Baselines auf verschiedenen Repräsentationen: cleaned vs. semantisch (PCA/no PCA) vs. fastText (PCA30/100) vs. enhanced (PCA/no PCA) vs. enhanced+semantisch
- Alignment über `row_id`, gemeinsamer 70/30-Split; feste Detektor-Params (fairer Vergleich)

In [9]:
import time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, auc
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

## Repräsentationen laden (indexiert über row_id)
- Label aus cleaned (Outlier = `fraudulent == 1`); enhanced+semantisch = PCA30-Konkatenation

In [10]:
LABEL = "fraudulent"
ds = "fake_jobs"

def_load = lambda name: pd.read_csv(f"../../data/preprocessed/{name}_{ds}.csv").set_index("row_id")
cleaned = def_load("cleaned")
semantic_pca100 = def_load("semantic_pca100")
semantic_pca = def_load("semantic_pca30")
fast_text_pca100 = def_load("fast_text_pca100")
fast_text_pca30 = def_load("fast_text_pca30")
enhanced = def_load("enhanced")
enhanced_pca = def_load("enhanced_pca30")

reps = {
    "cleaned": cleaned.drop(columns=[LABEL]),
    "semantic_pca100": semantic_pca100.drop(columns=[LABEL]),
    "semantic_pca30": semantic_pca.drop(columns=[LABEL]),
    "fast_text_pca100": fast_text_pca100.drop(columns=[LABEL]),
    "fast_text_pca30": fast_text_pca30.drop(columns=[LABEL]),
    "enhanced": enhanced.drop(columns=[LABEL]),
    "enhanced_pca30": enhanced_pca.drop(columns=[LABEL]),
    "enhanced_semantic_pca30": enhanced_pca.drop(columns=[LABEL]).join(
        semantic_pca.drop(columns=[LABEL]), how="inner", lsuffix="_enh", rsuffix="_sem"),
}

## Gemeinsamer Index & Split
- Schnittmenge aller Repräsentationen (robust gegen unvollständige semantic-CSVs)

In [11]:
common = cleaned.index
for r in reps.values():
    common = common.intersection(r.index)
common = common.sort_values()
y = cleaned.loc[common, LABEL].values
print("common rows:", len(common), "outlier rate", round(y.mean(), 4))

tr_id, te_id = train_test_split(common, test_size=0.3, stratify=y, random_state=42)
y_train = cleaned.loc[tr_id, LABEL].values
y_test = cleaned.loc[te_id, LABEL].values

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("fake_jobs_experiment_2")

common rows: 17880 outlier rate 0.0484


<Experiment: artifact_location='file:///home/debian/TFM_master_thesis/fake_job_notebooks/exp2/../../mlruns/598453087861783542', creation_time=1782730849067, experiment_id='598453087861783542', last_update_time=1782730849067, lifecycle_stage='active', name='fake_jobs_experiment_2', tags={}, trace_location=None, workspace='default'>

## Detektoren x Repräsentationen
- Beste Hyperparameter aus Exp 1 (README), **kein GridSearch**; AutoEncoder unsupervised (Originalverteilung, GPU)

In [12]:
# beste Hyperparameter aus Experiment 1 (README) — kein GridSearch in Exp 2
detectors = {
    "iforest": (IForest, {"n_estimators": 100, "max_features": 1.0, "random_state": 42}, False),
    "loda": (LODA, {"n_bins": 20, "n_random_cuts": 200}, False),
    "ecod": (ECOD, {}, False),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [64, 32], "epoch_num": 20, "random_state": 42, "device": "cuda"}, False),
}

for rep_name, rep in reps.items():
    Xtr = rep.loc[tr_id].values
    Xte = rep.loc[te_id].values
    for det_name, (Model, params, inlier_only) in detectors.items():
        t0 = time.perf_counter()
        Xfit = Xtr[y_train == 0] if inlier_only else Xtr
        model = Model(**params)
        model.fit(Xfit)
        scores = model.decision_function(Xte)
        runtime = time.perf_counter() - t0
        ap = average_precision_score(y_test, scores)
        prec, rec, _ = precision_recall_curve(y_test, scores)
        auprc = auc(rec, prec)
        auroc = roc_auc_score(y_test, scores)
        with mlflow.start_run(run_name=f"{rep_name}__{det_name}"):
            mlflow.log_param("representation", rep_name)
            mlflow.log_param("detector", det_name)
            mlflow.log_param("n_features", rep.shape[1])
            mlflow.log_metric("average_precision", ap)
            mlflow.log_metric("auprc", auprc)
            mlflow.log_metric("auc_roc", auroc)
            mlflow.log_metric("runtime_s", runtime)
        print(f"{rep_name:24s} {det_name:12s} AP={ap:.4f} AUPRC={auprc:.4f} AUC={auroc:.4f} feat={rep.shape[1]} t={runtime:.1f}s")

cleaned                  iforest      AP=0.0876 AUPRC=0.0858 AUC=0.6141 feat=13 t=0.4s
cleaned                  loda         AP=0.0432 AUPRC=0.0428 AUC=0.4705 feat=13 t=0.2s
cleaned                  ecod         AP=0.0759 AUPRC=0.0745 AUC=0.6125 feat=13 t=0.1s


Training: 100%|██████████| 20/20 [00:21<00:00,  1.06s/it]


cleaned                  autoencoder  AP=0.0838 AUPRC=0.0816 AUC=0.6003 feat=13 t=21.8s
semantic_pca100          iforest      AP=0.0581 AUPRC=0.0569 AUC=0.5390 feat=139 t=0.4s
semantic_pca100          loda         AP=0.0452 AUPRC=0.0448 AUC=0.4832 feat=139 t=0.3s
semantic_pca100          ecod         AP=0.0460 AUPRC=0.0457 AUC=0.4903 feat=139 t=1.0s


Training: 100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


semantic_pca100          autoencoder  AP=0.0584 AUPRC=0.0574 AUC=0.5284 feat=139 t=22.1s
semantic_pca30           iforest      AP=0.0869 AUPRC=0.0842 AUC=0.6197 feat=69 t=0.3s
semantic_pca30           loda         AP=0.0387 AUPRC=0.0384 AUC=0.4303 feat=69 t=0.3s
semantic_pca30           ecod         AP=0.0587 AUPRC=0.0580 AUC=0.5503 feat=69 t=0.4s


...

enhanced                 autoencoder  AP=0.1822 AUPRC=0.1793 AUC=0.8102 feat=512 t=23.4s
enhanced_pca30           iforest      AP=0.1496 AUPRC=0.1453 AUC=0.7071 feat=30 t=0.3s
enhanced_pca30           loda         AP=0.1237 AUPRC=0.1208 AUC=0.7154 feat=30 t=0.2s
enhanced_pca30           ecod         AP=0.1100 AUPRC=0.1074 AUC=0.6975 feat=30 t=0.2s


Training: 100%|██████████| 20/20 [00:21<00:00,  1.05s/it]


enhanced_pca30           autoencoder  AP=0.0863 AUPRC=0.0849 AUC=0.6453 feat=30 t=21.8s
enhanced_semantic_pca30  iforest      AP=0.1136 AUPRC=0.1119 AUC=0.6781 feat=99 t=0.3s
enhanced_semantic_pca30  loda         AP=0.0448 AUPRC=0.0445 AUC=0.5056 feat=99 t=0.2s
enhanced_semantic_pca30  ecod         AP=0.0825 AUPRC=0.0814 AUC=0.6579 feat=99 t=0.7s


Training: 100%|██████████| 20/20 [00:20<00:00,  1.04s/it]


enhanced_semantic_pca30  autoencoder  AP=0.0936 AUPRC=0.0919 AUC=0.6777 feat=99 t=21.5s
